In [1]:
# ============================================================
# CANDIDATE ELIMINATION ALGORITHM
# CAR EVALUATION DATASET
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
from itertools import product


# ============================================================
# 2. LOAD CAR EVALUATION DATASET
# ============================================================

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

columns = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "class"
]

df = pd.read_csv(
    url,
    names=columns
)


# ============================================================
# 3. DATASET INFORMATION
# ============================================================

TARGET_ATTR = "class"

ATTRIBUTES = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety"
]


print("=" * 70)
print("CAR EVALUATION DATASET")
print("=" * 70)

print(df.head(10))

print("\nDataset shape:", df.shape)

print("\nClass distribution:")
print(df["class"].value_counts())


# ============================================================
# 4. USE TWO CLASSES FOR CANDIDATE ELIMINATION
# ============================================================
#
# Candidate Elimination is a binary concept-learning algorithm.
#
# Therefore, we define:
#
#     Positive = good
#     Negative = not good
#
# The remaining classes:
#     unacc
#     acc
#     vgood
#
# are treated as negative examples.
# ============================================================

POSITIVE_CLASS = "good"


# ============================================================
# 5. DOMAIN OF EACH ATTRIBUTE
# ============================================================

DOMAINS = {
    attr: sorted(df[attr].unique())
    for attr in ATTRIBUTES
}


print("\nAttribute Domains:")

for attr in ATTRIBUTES:
    print(
        f"{attr}: {DOMAINS[attr]}"
    )


# ============================================================
# 6. HYPOTHESIS REPRESENTATION
# ============================================================
#
# A hypothesis is represented as:
#
#     value       -> exact value
#     '?'         -> any value
#     'Ø'         -> no value / most specific
#
# Example:
#
# ('low', '?', '?', '?', '?', 'high')
#
# means:
#
# buying = low
# safety = high
# all other attributes = anything
# ============================================================

NULL = "Ø"
ANY = "?"


# ============================================================
# 7. CHECK WHETHER A HYPOTHESIS COVERS AN EXAMPLE
# ============================================================

def covers(hypothesis, example):

    for h, value in zip(
        hypothesis,
        example
    ):

        # NULL means no example can satisfy it
        if h == NULL:
            return False

        # ? matches any value
        if h == ANY:
            continue

        # Exact value must match
        if h != value:
            return False

    return True


# ============================================================
# 8. CHECK CONSISTENCY
# ============================================================

def consistent_with(
    hypothesis,
    example,
    label
):

    covered = covers(
        hypothesis,
        example
    )

    if label == "positive":

        return covered

    else:

        return not covered


# ============================================================
# 9. GENERALITY CHECK
# ============================================================

def more_general_or_equal(
    h1,
    h2
):

    """
    Returns True if h1 is more general
    than or equal to h2.
    """

    for a, b in zip(h1, h2):

        # ? is more general than everything
        if a == ANY:

            continue

        # Ø is only more general than Ø
        if a == NULL:

            if b != NULL:
                return False

            continue

        # Exact value
        if b == NULL:

            continue

        if a != b:

            return False

    return True


# ============================================================
# 10. MINIMAL GENERALIZATIONS
# ============================================================

def minimal_generalizations(
    hypothesis,
    example
):

    h = list(hypothesis)

    for i, value in enumerate(example):

        if h[i] == NULL:

            h[i] = value

        elif h[i] != value:

            h[i] = ANY

    return {tuple(h)}


# ============================================================
# 11. MINIMAL SPECIALIZATIONS
# ============================================================

def minimal_specializations(
    hypothesis,
    example
):

    specializations = set()

    for i, value in enumerate(hypothesis):

        # Only ? can be specialized
        if value != ANY:
            continue

        for domain_value in DOMAINS[
            ATTRIBUTES[i]
        ]:

            # Do not choose the value
            # present in the negative example
            if domain_value != example[i]:

                new_hypothesis = list(
                    hypothesis
                )

                new_hypothesis[i] = (
                    domain_value
                )

                specializations.add(
                    tuple(new_hypothesis)
                )

    return specializations


# ============================================================
# 12. REMOVE MORE GENERAL HYPOTHESES
# ============================================================

def remove_more_general(
    boundary
):

    boundary = list(boundary)

    result = []

    for h in boundary:

        if not any(
            h != other
            and more_general_or_equal(
                h,
                other
            )
            for other in boundary
        ):

            result.append(h)

    return set(result)


# ============================================================
# 13. REMOVE MORE SPECIFIC HYPOTHESES
# ============================================================

def remove_more_specific(
    boundary
):

    boundary = list(boundary)

    result = []

    for h in boundary:

        if not any(
            h != other
            and more_general_or_equal(
                other,
                h
            )
            for other in boundary
        ):

            result.append(h)

    return set(result)


# ============================================================
# 14. HYPOTHESIS TO STRING
# ============================================================

def hypothesis_to_string(
    hypothesis
):

    conditions = []

    for attr, value in zip(
        ATTRIBUTES,
        hypothesis
    ):

        conditions.append(
            f"{attr}={value}"
        )

    return "(" + ", ".join(
        conditions
    ) + ")"


# ============================================================
# 15. INITIAL BOUNDARIES
# ============================================================

num_attributes = len(
    ATTRIBUTES
)


# Most specific hypothesis
S = {
    tuple(
        [NULL] * num_attributes
    )
}


# Most general hypothesis
G = {
    tuple(
        [ANY] * num_attributes
    )
}


# ============================================================
# 16. CANDIDATE ELIMINATION
# ============================================================

def candidate_elimination(
    training_data
):

    # --------------------------------------------------------
    # Initialize S and G
    # --------------------------------------------------------

    S = {
        tuple(
            [NULL] * num_attributes
        )
    }

    G = {
        tuple(
            [ANY] * num_attributes
        )
    }


    # --------------------------------------------------------
    # Process examples one by one
    # --------------------------------------------------------

    for index, row in training_data.iterrows():

        example = tuple(
            row[attr]
            for attr in ATTRIBUTES
        )


        # ----------------------------------------------------
        # Positive example
        # ----------------------------------------------------

        if row[TARGET_ATTR] == POSITIVE_CLASS:

            print(
                f"\nExample {index + 1}: "
                f"POSITIVE"
            )

            print(
                "Example:",
                example
            )


            # Remove hypotheses from G
            # that do not cover positive example

            G = {
                g
                for g in G
                if covers(
                    g,
                    example
                )
            }


            new_S = set()


            for s in S:

                if covers(
                    s,
                    example
                ):

                    new_S.add(s)

                else:

                    generalizations = (
                        minimal_generalizations(
                            s,
                            example
                        )
                    )


                    for h in generalizations:

                        # h must be no more general
                        # than every member of G

                        if any(
                            more_general_or_equal(
                                h,
                                g
                            )
                            for g in G
                        ):

                            new_S.add(h)


            S = new_S


        # ----------------------------------------------------
        # Negative example
        # ----------------------------------------------------

        else:

            print(
                f"\nExample {index + 1}: "
                f"NEGATIVE"
            )

            print(
                "Example:",
                example
            )


            # Remove hypotheses from S
            # that cover negative example

            S = {
                s
                for s in S
                if not covers(
                    s,
                    example
                )
            }


            new_G = set()


            for g in G:

                if not covers(
                    g,
                    example
                ):

                    new_G.add(g)

                else:

                    specializations = (
                        minimal_specializations(
                            g,
                            example
                        )
                    )


                    for h in specializations:

                        # h must be more general
                        # than some member of S

                        if any(
                            more_general_or_equal(
                                h,
                                s
                            )
                            for s in S
                        ):

                            new_G.add(h)


            G = new_G


        # ----------------------------------------------------
        # Remove redundant hypotheses
        # ----------------------------------------------------

        S = remove_more_specific(
            S
        )

        G = remove_more_general(
            G
        )


        # ----------------------------------------------------
        # Display boundaries
        # ----------------------------------------------------

        print(
            "Specific Boundary (S):"
        )

        for s in S:

            print(
                " ",
                hypothesis_to_string(s)
            )


        print(
            "General Boundary (G):"
        )

        for g in G:

            print(
                " ",
                hypothesis_to_string(g)
            )


    return S, G


# ============================================================
# 17. RUN CANDIDATE ELIMINATION
# ============================================================

print("\n")
print("=" * 70)
print("CANDIDATE ELIMINATION ALGORITHM")
print("=" * 70)


# ------------------------------------------------------------
# IMPORTANT:
# Use a manageable subset for displaying the version-space
# calculations.
#
# The complete Car Evaluation dataset contains 1728 examples.
# Processing all 1728 examples while printing S and G after
# every example creates extremely large output.
# ------------------------------------------------------------

training_data = df.copy()


S_final, G_final = candidate_elimination(
    training_data
)


# ============================================================
# 18. FINAL SPECIFIC BOUNDARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL SPECIFIC BOUNDARY (S)")
print("=" * 70)


for s in S_final:

    print(
        hypothesis_to_string(s)
    )


# ============================================================
# 19. FINAL GENERAL BOUNDARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL GENERAL BOUNDARY (G)")
print("=" * 70)


for g in G_final:

    print(
        hypothesis_to_string(g)
    )


# ============================================================
# 20. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("CANDIDATE ELIMINATION COMPLETED")
print("=" * 70)

print(
    "\nPositive class:",
    POSITIVE_CLASS
)

print(
    "Number of hypotheses in S:",
    len(S_final)
)

print(
    "Number of hypotheses in G:",
    len(G_final)
)

Streaming output truncated to the last 5000 lines.
  (buying=low, maint=low, doors=5more, persons=more, lug_boot=med, safety=med)
  (buying=low, maint=low, doors=4, persons=4, lug_boot=big, safety=high)
  (buying=low, maint=high, doors=4, persons=more, lug_boot=big, safety=med)
  (buying=med, maint=low, doors=4, persons=4, lug_boot=med, safety=high)
  (buying=med, maint=low, doors=3, persons=more, lug_boot=big, safety=high)
  (buying=low, maint=high, doors=4, persons=4, lug_boot=med, safety=med)
  (buying=low, maint=high, doors=3, persons=more, lug_boot=big, safety=med)
  (buying=low, maint=med, doors=4, persons=more, lug_boot=med, safety=high)
  (buying=med, maint=low, doors=3, persons=4, lug_boot=med, safety=high)
  (buying=low, maint=med, doors=5more, persons=4, lug_boot=med, safety=high)
  (buying=low, maint=med, doors=3, persons=4, lug_boot=med, safety=med)
  (buying=low, maint=low, doors=5more, persons=more, lug_boot=big, safety=med)
  (buying=low, maint=med, doors=5more, persons